# PHASE 3 — Ensemble AdaBoost và so sánh toàn bộ

> **Đồ án thực hành giữa kỳ — Nhận dạng chữ số viết tay MNIST**  
> **Vai trò:** Mô hình bắt buộc thứ 4 + kết luận và so sánh đồ án  
> **Mô hình trong phase:** AdaBoost-SAMME-NP; tổng hợp NB + Logistic + MLP (+ CNN)  
> **Tình trạng:** *Bộ khung giao việc; toàn bộ notebook chỉ có ô Markdown TODO, chưa có code hoặc số liệu.*

**Tài liệu bắt buộc:** `DoAn_ThucHanh_NhanDang.pdf` (Đồ án 1, trang 1–2; phụ lục trang 4).  
**Tham khảo:** `BTVN_Lab01.ipynb`, `NaiveBayes(1).ipynb`, `TL_Lab03.ipynb`.  
**Kế hoạch tổng:** `plan.md`.

**Luồng áp dụng:** **DATA → X, y → split → preprocessing → fit → predict → metrics → visualize → analyze** (theo `BTVN_Lab01.ipynb`).

---

## 0. Mục tiêu và quy ước

| Bước | Nội dung | Đầu ra tối thiểu |
|---|---|---|
| 1 | MNIST/split và chuẩn hóa | 54k/6k/10k, 784 chiều, seed 42 |
| 2 | Tự cài Ensemble | AdaBoost SAMME đa lớp và decision stump |
| 3 | Thí nghiệm siêu tham số | ≥1 biểu đồ theo `n_estimators`, chọn bằng validation |
| 4 | Đánh giá mô hình cuối | **Train/validation/test**: 4 metrics + CM, test sau khi khóa tham số |
| 5 | So sánh 4 phương pháp chính | NB / Logistic / MLP / Ensemble |
| 6 | Phụ lục CNN và kết luận | Bảng CNN + trade-off + artifact |

**Lưu ý theo PDF:** 4 loại mô hình **bắt buộc** phải tự cài phần lõi; sklearn chỉ **đối chiếu**. CNN là phần nâng cao, bổ sung thành mô hình thứ 5 theo yêu cầu hiện tại. Notebook này không huấn luyện lại ba mô hình khác nếu đã có kết quả chuẩn; chỉ tổng hợp CSV đồng nhất (có thể chạy lại nếu thiếu tệp).

**Tái sử dụng format:** `NaiveBayes(1).ipynb` (6 bước + TODO), `BTVN_Lab01.ipynb` (metric), `TL_Lab03.ipynb` (`evaluate`/summary/time).

### 0.1. Cấu hình và kế thừa tiện ích

**TODO:**
1. Import NumPy, Matplotlib, pandas, time; `SEED=42`, nhãn 0..9, tạo thư mục `results/`, `figures/`, `models/`.
2. Dùng cùng hợp đồng CSV: `phase, model, split, accuracy, precision_macro, recall_macro, f1_macro, train_seconds, predict_seconds, n_samples, seed, params`.
3. Thống nhất tên model `AdaBoost-SAMME-NP`, cài kiểm tra môi trường/CPU/GPU và đo thời gian như Phase 1/2.
4. Chỉ import sklearn cho split/metric/đối chiếu, **không dùng làm Ensemble chính**.

## 1. Dữ liệu MNIST và đánh giá dùng chung

### 1.1. Nạp/kiểm MNIST và cùng split

**TODO:**
1. Tải **MNIST 28×28** train gốc 60k + test gốc 10k; **bắt buộc đọc** `artifacts/split_indices.npz` từ `phase1.ipynb`, áp `train_idx`, `val_idx`, `test_idx` theo **thứ tự đã lưu** → **train 54k, val 6k, test 10k**. Nếu thiếu/không hợp lệ, yêu cầu tạo từ Phase 1; **không chia lại chỉ bằng `SEED=42`**.
2. Tạo pixel `float32` trong [0,1] từ ảnh nguồn/255, flattened `(N,784)` cho decision stump/AdaBoost; giữ ảnh 28×28 để phân tích lỗi. Ghi rõ **pipeline đầu vào của AdaBoost**, không dùng nhầm scaler riêng của Logistic/MLP/CNN.
3. Trước khi áp chỉ số, xác thực `train_source_sha256`/`test_source_sha256` trên **ảnh + nhãn gốc theo đúng thứ tự**; kiểm shape, không trùng/mất mẫu, unique labels, class count phân tầng và ảnh mẫu khớp thứ tự.
4. **Không** lấy mẫu test để tập weak learner; không thay split để làm ensemble đẹp hơn.

**Tái sử dụng:** `phase1.ipynb` mục 1, `BTVN_Lab01.ipynb` quy trình DATA → analyze.

### 1.2. Kiểm soát leakage, runtime và metric chuẩn

**TODO:**
1. Chứng minh train/val/test dùng **đúng cùng `split_indices.npz` và hash ảnh/nhãn gốc** như Phase 1/2: nhãn, seed, thứ tự mẫu; kiểm pipeline tiền xử lý riêng của từng mô hình, không fit preprocessing trên val/test.
2. Xây hàm đánh giá chung trả accuracy, `precision_macro`, `recall_macro`, `f1_macro`, confusion matrix **10×10** và `predict_seconds`.
3. Định nghĩa rõ `train_seconds`: riêng fit AdaBoost (không bao gồm load/EDA); cùng cách đo với Phase 1/2.
4. Để kiểm tốc độ, thử trên subset **của train** ban đầu; báo cáo rõ nếu thí nghiệm cuối không train full 54k.

**Tái sử dụng:** `TL_Lab03.ipynb` phần 3.3 (`evaluate`, `summarize`, `time.perf_counter`).

## 2. Ensemble bắt buộc — AdaBoost đa lớp SAMME tự cài đặt

### 2.1. Giải thích Ensemble/AdaBoost và công thức SAMME

**TODO:**
1. Giới thiệu boosting: gán trọng số theo độ khó, kết hợp nhiều weak learners để tạo strong learner.
2. Chọn **decision stump (cây một tầng)** làm weak learner; nêu lỗi phân lớp có trọng số `err_t`.
3. Vì MNIST có **K=10 lớp**, dùng **SAMME đa lớp**, không thay bằng cập nhật chỉ đúng cho AdaBoost nhị phân. Viết công thức trọng số learner `alpha_t = learning_rate * [ln((1-err_t)/err_t) + ln(K-1)]` và mẫu sai được tăng trọng số.
4. Viết rule predict `argmax_k sum_t alpha_t * 1[h_t(x)=k]`; trình bày điều kiện chấp nhận weak learner khi `err_t < 1 - 1/K`.

**Sản phẩm:** công thức + pseudo-code Ensemble cho báo cáo.

### 2.2. Tự cài `DecisionStump` có trọng số

**TODO:**
1. Với `X` shape `(N,784)` và sample weights, tìm `feature`, `threshold` và dự đoán lớp của 2 nhánh làm nhỏ nhất **tổng lỗi phân loại có trọng số**.
2. Tự cài `fit(X,y,sample_weight)` và `predict(X)`; xử lý tie, nhãn 0..9 và pixel có giá trị bằng nhau.
3. Tối ưu hợp lý cho MNIST: giới hạn candidate thresholds, feature subsampling có seed, vectorize khi có thể; ghi rõ chiến lược để kết quả tái lập.
4. Unit test trên vài mẫu/feature nhỏ và kiểm weighted error khớp tính tay.

**Điểm cốt lõi:** không tạo `DecisionTreeClassifier` sklearn làm weak learner của lời giải chính.

### 2.3. Tự cài vòng lặp `AdaBoostSAMME.fit`

**TODO:**
1. Khởi tạo `sample_weight = 1/N`, lập danh sách `stumps_`, `alphas_`.
2. Mỗi vòng: fit stump với sample weights → weighted error → tính `alpha_t` đúng K=10 → cập nhật/trọng số hóa mẫu phân lớp sai → normalize weights.
3. Xử lý epsilon nhỏ/0, weak learner không tốt hơn ngưỡng `1−1/K`, tránh NaN/Inf/tràn số.
4. Lưu `n_estimators`, `learning_rate`, seed, lịch sử lỗi/alpha/trọng số để có thể giải thích/biểu đồ.
5. Kiểm vòng lặp tăng số stump, không dùng nhãn validation/test khi cập nhật trọng số.

**Sản phẩm:** lớp AdaBoost SAMME tự viết `fit/predict`.

### 2.4. `predict`, `decision_function` và kiểm tra tính đúng

**TODO:**
1. Cài scores 10 lớp bằng tổng `alpha` cho lớp dự đoán bởi từng weak learner.
2. `predict` trả label 0..9; kiểm output shape, finite values, handling ties.
3. Test chạy thử 1/3/5 stump và xem lỗi trên **train subset** có hợp lý; đánh giá `fit` không thay đổi tập val/test.
4. Kiểm ảnh hưởng số weak learners đến fitting/memory; ghi thời gian mỗi vòng nếu cần.

**Lưu ý:** chỉ kết luận sau khi thực nghiệm, không ghi trước ensemble chắc chắn tốt hơn NB/LR.

### 2.5. Kiểm chứng bằng sklearn (chỉ đối chiếu)

**TODO:**
1. Thiết lập đối chiếu với `sklearn.ensemble.AdaBoostClassifier` dùng decision stump khi phiên bản thư viện hỗ trợ đúng SAMME; ghi phiên bản và các khác biệt API/thuật toán.
2. Đặt cùng split/seed/số weak learners, so trên **validation** về accuracy/F1 macro và thời gian; không kỳ vọng khớp chính xác nếu chiến lược threshold/feature subsampling khác.
3. Báo sai lệch và nguyên nhân: cách chọn stump, threshold, stopping, smoothing, số cây.
4. Mô hình **tự cài** mới là dòng `AdaBoost-SAMME-NP` để so 4 phương pháp; không thay bằng sklearn.

**Tham khảo phong cách:** `TL_Lab03.ipynb` phần **4. Đối chiếu với scikit-learn**.

## 3. Thí nghiệm siêu tham số, chọn mô hình và đánh giá cuối

### 3.1. Ảnh hưởng của số weak learners (`n_estimators`)

**TODO:**
1. Thử **ít nhất 3 mức** số vòng boosting, ví dụ `10, 30, 60` (điều chỉnh theo thời gian thực thi); cố định learning rate, stump depth, seed và split.
2. Mỗi cấu hình chỉ **fit trên train** và **chọn theo validation**, thu `val_f1_macro`, `val_accuracy`, `train_seconds`.
3. Vẽ đồ thị chính `n_estimators → validation F1 macro` (hoặc accuracy) với tên/nhãn trục, điểm dữ liệu; có thể thêm đồ thị `n_estimators → train_seconds`.
4. Giải thích đánh đổi độ chính xác/thời gian/overfit dựa trên số liệu thực, không tuning theo test.

**Sản phẩm bắt buộc:** ≥1 hình so sánh nhiều giá trị hyperparameter Phase 3.

### 3.2. Thí nghiệm mở rộng (nếu đủ thời gian)

**TODO — tùy chọn:**
1. Khảo sát `learning_rate` (ví dụ `0.3, 0.7, 1.0`) hoặc ngưỡng/feature subsampling với `n_estimators` cố định.
2. Chỉ dùng validation để chọn, vẽ biểu đồ có nhãn và giải thích nguyên nhân; ghi độ ổn định theo seed nếu có.
3. Không mở rộng khiến bỏ sót bảng so sánh 4 phương pháp bắt buộc.

### 3.3. Đánh giá Ensemble — train / validation / test

**TODO:**
1. **Sau thí nghiệm 3.1 (và 3.2 nếu có)**, khóa `n_estimators`/siêu tham số theo validation; fit AdaBoost trên train rồi mới đánh giá **accuracy, precision_macro, recall_macro, f1_macro** cho **cả 3 split**, test chỉ dùng lúc đánh giá cuối.
2. Vẽ **3 confusion matrix 10×10** train/validation/test có label 0..9; thêm chuẩn hóa theo hàng test nếu cần.
3. Đo riêng fit time và predict time trên 3 split; báo lỗi per-digit, ảnh dự đoán sai, top 3 cặp nhầm.
4. Phân tích chênh lệch train/validation/test, độ phức tạp/số stump, giải thích stump feature nào quan trọng.

**Tái sử dụng:** công thức macro trong `BTVN_Lab01.ipynb`, ô lỗi của `NaiveBayes(1).ipynb` mục 2.2.6.

## 4. Tổng hợp 4 phương pháp bắt buộc + CNN nâng cao

### 4.1. Đọc kết quả của Phase 1 và Phase 2

**TODO:**
1. Đọc `results/phase1_metrics.csv` (GaussianNB-NP, LogReg-GD-NP) và `results/phase2_metrics.csv` (MLP, CNN) cùng CSV Ensemble của phase này.
2. Kiểm 5 nhãn model đúng, split trong `{train, validation, test}`, các cột metric đúng kiểu, không NaN; **đối chiếu `split_indices.npz`, hash nguồn, thứ tự mẫu và pipeline tiền xử lý của từng phase (không chỉ SEED)**. Nếu thiếu CSV/metadata thì hướng dẫn chạy lại phase tương ứng.
3. **Không** tự sinh số liệu thay thế; không copy giả accuracy từ nguồn khác.
4. Tạo bảng kết quả 5 mô hình trên ba tập để kiểm độ nhất quán, sau đó tách bảng theo rubric.

### 4.2. Bảng so sánh CHÍNH — đúng 4 phương pháp PDF

**TODO:**
1. Chỉ lấy **4 hàng model**: `GaussianNB-NP`, `LogReg-GD-NP`, `MLP`, `AdaBoost-SAMME-NP` (test đã cố định best model theo validation).
2. Các cột: `accuracy`, `precision_macro`, `recall_macro`, `f1_macro`, `train_seconds`, `predict_seconds`, `ability_to_explain` (định tính, nêu nguồn lập luận).
3. Tạo thêm bảng 4 mô hình × **train/validation/test** hoặc pivot metrics để phát hiện overfit/underfit.
4. So sánh trên cùng số lượng mẫu và ghi nền tảng phần cứng khi đo timing.

**Bắt buộc trong báo cáo:** PDF yêu cầu **so sánh 4 phương pháp**, không thay bảng này bằng bảng chỉ MLP vs CNN.

### 4.3. Bảng bổ sung — CNN là phương pháp thứ 5

**TODO:**
1. Thêm `CNN` vào bảng mở rộng (5 models) và mô tả đây là **yêu cầu nâng cao PDF / yêu cầu của kế hoạch Phase 2**.
2. Trích kết quả MLP vs CNN từ Phase 2: train/validation/test accuracy và F1 macro, train/predict time, số tham số/chi phí.
3. Ghi rõ nếu CNN dùng GPU còn mô hình NumPy dùng CPU thì thời gian thô **không công bằng**, không kết luận cường độ tính toán chỉ dựa vào giây đo được.

### 4.4. Biểu đồ so sánh chất lượng và thời gian

**TODO:**
1. Vẽ bar chart accuracy/F1 macro test của **4 mô hình chính** và chú giải CNN riêng nếu cần.
2. Vẽ biểu đồ train_seconds / predict_seconds theo mô hình (tách ra hoặc dùng log-scale nếu chênh nhiều), ghi phần cứng và số mẫu.
3. Đảm bảo nhãn rõ, không đặt hai đơn vị khác nhau vào cùng một trục khó diễn giải.
4. Mô tả các xu hướng thực tế dựa trên bảng/biểu đồ.

### 4.5. Phân tích ưu / nhược điểm và khả năng giải thích

**TODO:**
1. GaussianNB: giả định pixel độc lập, `priors/mean/variance` dễ đọc, tốc độ; so với mức độ chính xác đo được.
2. Logistic GD: trọng số tuyến tính dễ giải thích, tốc độ hội tụ phụ thuộc learning rate; so với các đường loss/thời gian thực tế.
3. MLP: ranh giới phi tuyến, số tham số, khả năng overfit; interpretability hạn chế hơn mô hình đơn giản.
4. AdaBoost: kết hợp stump, số vòng ảnh hưởng fit/predict time và lỗi, giải thích qua weak learner/feature importance.
5. CNN: tận dụng không gian ảnh, chiều sâu/features học được, chi phí tính toán và giải thích.

**Yêu cầu của PDF:** nêu riêng **độ chính xác, thời gian huấn luyện, thời gian dự đoán, khả năng giải thích**, kèm dẫn chứng trong bảng.

### 4.6. Thảo luận confusion matrix và lỗi tổng thể

**TODO:**
1. Với 4 mô hình chính (và CNN), xác định cặp chữ số dễ nhầm qua confusion matrix và minh họa ảnh.
2. Giải thích khác biệt theo giả định/đặc trưng kiến trúc, nhưng phân biệt giả thuyết với kết quả đo.
3. Thảo luận train–val/test gap và sai khác fit/predict time, nhất là CPU/GPU.
4. Nêu hạn chế: chi phí tự cài đặt, tập dữ liệu chỉ MNIST, ảnh số bị lệch/nhòe, giới hạn siêu tham số.

### 4.7. Xuất bảng tổng hợp và kết luận đồ án

**TODO:**
1. Lưu `results/phase3_metrics.csv` (3 hàng AdaBoost × train/validation/test) và `results/comparison.csv` (ít nhất 4 model chính; có thể lưu bảng mở rộng CNN riêng).
2. Lưu biểu đồ n_estimators, confusion matrix, so sánh accuracy/F1/timing và config tự cài.
3. Soạn bullet có số liệu cho báo cáo PDF 10–15 trang: dữ liệu, công thức/pseudocode, thiết kế, thí nghiệm, kết quả, thảo luận, kết luận, hạn chế.
4. Tạo checklist nguồn tham khảo/AI, phân công nhóm và phần trăm đóng góp (điền theo thực tế), chuẩn bị slide 10–15 phút.
5. Kiểm schema/tệp Phase 1,2,3 có thể tái lập khi chạy lại.

---

## 5. Câu hỏi thảo luận và kiểm thử trước nộp

**TODO:**
1. Vì sao với K=10 lớp, không thể dùng nguyên alpha/cập nhật trọng số AdaBoost nhị phân?
2. Decision stump có thể chậm/thiếu sức phân biệt trên ảnh 784 chiều ở bước nào? Nhóm khắc phục ra sao mà không thay lõi bằng sklearn?
3. Mô hình nào tốt nhất theo test F1 macro? Có phải mô hình nhanh nhất khi dự đoán không? Dẫn số liệu.
4. Sự khác nhau giữa bảng **4 phương pháp theo rubric** và bảng **5 mô hình có CNN**?
5. Kiểm `Restart & Run All`, không placeholder/`NotImplementedError`, không dùng data leakage, không giả số liệu, khai báo nguồn và AI.

**DONE:** [ ] AdaBoost tự cài; [ ] đủ metrics/CM 3 split; [ ] ≥1 hình siêu tham số; [ ] bảng 4 phương pháp và CNN phụ; [ ] 4 tiêu chí so sánh; [ ] CSV/plots + thảo luận.